In [ ]:
#############################################
###  Phase 1. Shapiro's test and QQ-plot  ###
#############################################

import pandas as pd
import numpy as np
from scipy.stats import shapiro, probplot
import matplotlib.pyplot as plt
import os

output_dir = './'
base_plot_dir = os.path.join(output_dir, './qq_plots')

if not os.path.exists(base_plot_dir):
    os.makedirs(base_plot_dir)

df = pd.read_excel('./HarvestData.xlsx') 

# select the analyzing factors (split first separately)
traits = df.columns[5:]
genotypes = df['Genotype'].unique()
treatments = df['Treatment'].unique()

detailed_results = []

print("Normality testing by Genotype x Treatment and QQ plots")

for trait in traits:

    safe_trait_name = trait.replace('/', '_')
    
    #split the folder in each factor (since the files are a lot)
    trait_plot_dir = os.path.join(base_plot_dir, safe_trait_name)
    if not os.path.exists(trait_plot_dir):
        os.makedirs(trait_plot_dir)
        
    for g in genotypes:
        for t in treatments:
            #extract the data of certain combination of genotype and treatment and remove N/A
            subset = df[(df['Genotype'] == g) & (df['Treatment'] == t)][trait].dropna()
            
            if len(subset) >= 3:
                stat, p_val = shapiro(subset)
                res_status = 'Pass' if p_val > 0.05 else 'Fail'
                
                #QQ Plot generation
                plt.figure(figsize=(6, 5))
                probplot(subset, plot=plt)
                plt.title(f"QQ Plot: {trait}\nG:{g} | T:{t}\n(p-value: {p_val:.4f})")
                plt.grid(True, linestyle=':', alpha=0.6)
                
                # save in QQ_Genotype_Treatment.png
                plot_filename = f"QQ_{g}_{t}.png"
                plt.savefig(os.path.join(trait_plot_dir, plot_filename))
                plt.close()
                
            else:
                stat, p_val = np.nan, np.nan
                res_status = 'Insufficient Data'
            
            detailed_results.append({
                'Trait': trait,
                'Genotype': g,
                'Treatment': t,
                'Sample_Size': len(subset),
                'Shapiro_p_value': round(p_val, 4) if not np.isnan(p_val) else np.nan,
                'Normality': res_status
            })

#save in csv
detailed_summary_df = pd.DataFrame(detailed_results)
save_path = os.path.join(output_dir, 'detailed_cell_normality.csv')
detailed_summary_df.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"\n[analysis completes]")
print(f"detailed csv saved in: {save_path}")
print(f"detailed qq plots saved in: {base_plot_dir}")

summary_table = detailed_summary_df.groupby('Trait')['Normality'].value_counts(normalize=True).unstack().fillna(0)
print("\nresults")
print(summary_table)

Normality testing by Genotype x Treatment and QQ plots


/mnt/c/Users/채은비/Desktop/Master thesis/venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:592: UserWarning: scipy.stats.shapiro: Input data has range zero. The results may not be accurate.
  res = hypotest_fun_out(*samples, **kwds)



[analysis completes]
detailed csv saved in: /mnt/c/Users/채은비/Desktop/Master thesis/03_Results/0301_statistics/detailed_cell_normality.csv
detailed qq plots saved in: /mnt/c/Users/채은비/Desktop/Master thesis/03_Results/0301_statistics/qq_plots

results
Normality                     Fail  Pass
Trait                                   
C/N                           0.10  0.90
C_Seeds                       0.10  0.90
FlagLeafLength                0.05  0.95
FlagLeafWeight_harvest        0.20  0.80
FlagLeafWidth                 0.10  0.90
FloweringDate                 0.70  0.30
HeadingDate                   0.70  0.30
MainPanicleNumber_Grains      0.10  0.90
MainPanicleWeight_Grains      0.00  1.00
MainPanicleWeight_dried       0.00  1.00
MainPanicleWeight_harvest     0.00  1.00
N_Seeds                       0.05  0.95
Number_Grains                 0.05  0.95
OLD_PanicleNumber_Ripe        0.10  0.90
PanicleCulmRatio              0.05  0.95
PanicleDensity                0.15  0.85
PanicleLeng

In [ ]:
###--- Filtering out inappropriate traits ---###

import pandas as pd
import os


base_dir = './'
input_file = os.path.join(base_dir, 'detailed_cell_normality.csv')
output_file = os.path.join(base_dir, 'filtered_detailed_cell_normality.csv')

df = pd.read_csv(input_file)

# calculate the ratio of fail (not normally distributed)
trait_fail_rates = df.groupby('Trait')['Normality'].apply(lambda x: (x == 'Fail').mean())

# filter the passed traits (normally distributed) and filtered out the failed traits
passed_traits = trait_fail_rates[trait_fail_rates < 0.5].index.tolist()
excluded_traits = trait_fail_rates[trait_fail_rates >= 0.5].index.tolist()

# extract only passed traits
filtered_df = df[df['Trait'].isin(passed_traits)]

# save the file
filtered_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"summarize the result")
print(f"previous trait number: {len(trait_fail_rates)}")
print(f"passed traits: {len(passed_traits)}")
print(f"excluded traits: {excluded_traits}")
print(f"\ncomplete: file has been saved: {output_file}")


if excluded_traits:
    print("\nratio of failed traits")
    print(trait_fail_rates[excluded_traits])

summarize the result
previous trait number: 45
passed traits: 42
excluded traits: ['FloweringDate', 'HeadingDate', 'RachisNodeNumber']

complete: file has been saved: /mnt/c/Users/채은비/Desktop/Master thesis/03_Results/0301_statistics/filtered_detailed_cell_normality.csv

ratio of failed traits
Trait
FloweringDate       0.70
HeadingDate         0.70
RachisNodeNumber    0.55
Name: Normality, dtype: float64


In [ ]:
#################################
###  Phase 1.2 Levene's test  ###
#################################

import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')


FILE_PATH     = "./HarvestData.xlsx"  
GENOTYPE_COL  = "Genotype"         
TREATMENT_COL = "Treatment"        

# Traits excluded according to the result of Shapiro-Wilk test
EXCLUDED_TRAITS = ['Give traits to exclude']

df = pd.read_excel(FILE_PATH)

print("list of column:", df.columns.tolist())
print("shape: {df.shape}")

all_traits = df.columns[5:].tolist()
trait_cols = [t for t in all_traits if t not in EXCLUDED_TRAITS]

print(f"excluded traits: {EXCLUDED_TRAITS}")
print(f"analyzing traits: {len(trait_cols)}")
print(f"Genotype: {sorted(df[GENOTYPE_COL].unique())}")
print(f"Treatment: {df[TREATMENT_COL].unique()}")

GENOTYPES  = sorted(df[GENOTYPE_COL].unique())
TREATMENTS = ["Trt1", "Trt2"]  



# Run standard levene's test (center = mean)

def run_levene(*groups, center='mean'):
    clean = [g.dropna().values for g in groups if len(g.dropna()) >= 2]
    if len(clean) < 2:
        return np.nan, np.nan
    stat, p = stats.levene(*clean, center=center)
    return round(stat, 4), round(p, 4)



# Levene 2-1
## test all genotype × treatment cells (center=mean): to check the assumption for execution of Two-way ANOVA

print("\nLevene 2-1 running...")

records_2_1 = []

for trait in trait_cols:
    groups, labels = [], []

    for geno in GENOTYPES:
        for trt in TREATMENTS:
            subset = df[
                (df[GENOTYPE_COL]  == geno) &
                (df[TREATMENT_COL] == trt)
            ][trait]
            if len(subset.dropna()) >= 2:
                groups.append(subset)
                labels.append(f"{geno}_{trt}")

    stat, p = run_levene(*groups, center='mean')

    records_2_1.append({
        "Trait"         : trait,
        "n_groups"      : len(groups),
        "Groups_tested" : ", ".join(labels),
        "Levene_stat"   : stat,
        "p_value"       : p,
        "Homogeneity"   : "PASS (p>=0.05)" if (not np.isnan(p) and p >= 0.05)
                          else "FAIL (p<0.05)"
    })

df_2_1 = pd.DataFrame(records_2_1)
df_2_1.to_csv("levene_2_1_results.csv", index=False)

print(df_2_1[["Trait", "n_groups", "Levene_stat", "p_value", "Homogeneity"]].to_string(index=False))
print(f"\nsaved as: levene_2_1_results.csv")
print(f"  Pass: {(df_2_1['Homogeneity'].str.startswith('PASS')).sum()}")
print(f"  Fail: {(df_2_1['Homogeneity'].str.startswith('FAIL')).sum()}")


# Levene 2-2
## test Trt1 vs Trt2 within same genotype (center=median, Brown-Forsythe): to check the variance distribution by treatment
### record the p-value and compare variance distribution between treatments (to flag)

print("\nLevene 2-2 running...")

records_2_2 = []

for trait in trait_cols:
    for geno in GENOTYPES:
        ctrl = df[
            (df[GENOTYPE_COL]  == geno) &
            (df[TREATMENT_COL] == "Trt1")
        ][trait]
        drgt = df[
            (df[GENOTYPE_COL]  == geno) &
            (df[TREATMENT_COL] == "Trt2")
        ][trait]

        stat, p = run_levene(ctrl, drgt, center='mean')

        # variance distribution comparison (treatments)
        var_ctrl = round(ctrl.dropna().var(), 4) if len(ctrl.dropna()) >= 2 else np.nan
        var_drgt = round(drgt.dropna().var(), 4) if len(drgt.dropna()) >= 2 else np.nan

        if not np.isnan(var_ctrl) and not np.isnan(var_drgt):
            direction = "Trt2 > Trt1" if var_drgt > var_ctrl else "Trt1 > Trt2"
        else:
            direction = "N/A"

        records_2_2.append({
            "Trait"              : trait,
            "Genotype"           : geno,
            "Var_Trt1"        : var_ctrl,
            "Var_Trt2"        : var_drgt,
            "Variance_direction" : direction,
            "Levene_stat"        : stat,
            "p_value"            : p,
            "Homogeneity"        : "PASS (p>=0.05)" if (not np.isnan(p) and p >= 0.05)
                                   else "FAIL (p<0.05)",
            "Note"               : "" if (np.isnan(p) or p >= 0.05)
                                   else f"Unequal variance: {direction}"
        })

df_2_2 = pd.DataFrame(records_2_2)
df_2_2.to_csv("levene_2_2_results.csv", index=False)

# only summarize the case of FAIL
fail_2_2 = df_2_2[df_2_2["Homogeneity"].str.startswith("FAIL")]
print(f"\nFAIL cases (p<0.05 Trait x Genotype combi):")
if len(fail_2_2) == 0:
    print("  -> no Fail: All met homogeneity")
else:
    print(fail_2_2[["Trait", "Genotype", "Var_Trt1", "Var_Trt2",
                     "Variance_direction", "p_value"]].to_string(index=False))

print(f"\nSaved as: levene_2_2_results.csv")
print(f"  Entire combi: {len(df_2_2)}  (Trait; {len(trait_cols)} x Genotype; {len(GENOTYPES)})")
print(f"  Fail combi: {len(fail_2_2)}")

list of column: ['Pot', 'Genotype', 'Treatment', 'Lane', 'Column', 'PlantHeight_PanicleTop', 'PlantHeight_PanicleBase', 'PlantHeight_Flagleaf', 'PlantHeight_TopNode', 'FlagLeafLength', 'FlagLeafWidth', 'OLD_PanicleNumber_Ripe', 'PanicleNumber_Ripe', 'PanicleNumber_Unripe', 'TillerNumber', 'RipePaniclesWeight_harvest', 'UnripePaniclesWeight_harvest', 'MainPanicleWeight_harvest', 'StrawWeight_harvest', 'FlagLeafWeight_harvest', 'UnripePaniclesWeight_dried', 'RipePaniclesWeight_dried', 'MainPanicleWeight_dried', 'StrawWeight_dried', 'SpikeletNumber', 'RachisNodeNumber', 'd13C_Flagleaf', 'd13C_Seeds', 'N_Seeds', 'C_Seeds', 'C/N', 'MainPanicleNumber_Grains', 'MainPanicleWeight_Grains', 'RipePaniclesNumber_Grains', 'RipePaniclesWeight_Grains', 'PanicleLength', 'PeduncleLength', 'ProductiveTillerRatio', 'PanicleDensity', 'PanicleCulmRatio', 'TotalWeight_dried', 'TotalWeight_harvest', 'Water_irrigated', 'WUE_drymatter', 'WUE_freshmatter', 'FloweringDate', 'HeadingDate', 'Number_Grains', 'Weigh